<a href="https://colab.research.google.com/github/aashutosh9901/InformationRetrieval/blob/main/Informationretrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Information Retrieval System

A simple Boolean Information Retrieval system built from scratch in Python: document collection → preprocessing → dictionary → inverted index → Boolean retrieval.

## 1. Import Libraries

Only Python's standard library is needed for this project.

In [ ]:
import re


## 2. Document Collection

Eight short documents on related IT/CS topics, stored as a dictionary keyed by document ID. Keeping the topics related on purpose creates realistic overlapping vocabulary (e.g. "machine", "learning", "data", "cloud") so Boolean queries return meaningful, non-trivial results.

In [ ]:
documents = {
    "D1": "Artificial intelligence is a branch of computer science that focuses on building systems "
          "capable of performing tasks that normally require human intelligence. These tasks include "
          "reasoning, problem solving, perception, and language understanding. Modern artificial "
          "intelligence systems often rely on machine learning techniques to improve their performance "
          "over time. Applications of artificial intelligence can be found in robotics, healthcare, "
          "finance, and many other fields.",

    "D2": "Machine learning is a subset of artificial intelligence that enables computer systems to "
          "learn patterns from data without being explicitly programmed. Machine learning algorithms "
          "build a model based on sample data, known as training data, in order to make predictions or "
          "decisions. Common types of machine learning include supervised learning, unsupervised "
          "learning, and reinforcement learning. Machine learning is widely used in recommendation "
          "systems, image recognition, and natural language processing.",

    "D3": "Data science is an interdisciplinary field that uses scientific methods, algorithms, and "
          "systems to extract knowledge and insights from structured and unstructured data. Data "
          "scientists often use statistics, programming, and domain knowledge to analyze large datasets. "
          "Data science combines elements of mathematics, computer science, and machine learning to "
          "solve real world problems. The growth of big data has increased demand for skilled data "
          "science professionals.",

    "D4": "Information retrieval is the process of obtaining relevant information from a large "
          "collection of documents based on a user query. Search engines are one of the most common "
          "applications of information retrieval. Information retrieval systems typically use "
          "techniques such as indexing, ranking, and query processing to return relevant results. "
          "Building an efficient inverted index is a core part of any information retrieval system.",

    "D5": "Natural language processing is a field of artificial intelligence concerned with the "
          "interaction between computers and human language. Natural language processing techniques "
          "allow computers to read, understand, and generate human language in a useful way. Common "
          "natural language processing tasks include text classification, sentiment analysis, and "
          "machine translation. Many natural language processing systems rely on machine learning "
          "models trained on large text datasets.",

    "D6": "A database is an organized collection of structured data that is stored and accessed "
          "electronically. Databases are managed using database management systems that allow users "
          "to create, read, update, and delete data efficiently. Relational databases store data in "
          "tables and use structured query language to manage the data. Databases are essential for "
          "storing information used by websites, applications, and information retrieval systems.",

    "D7": "Cloud computing is the delivery of computing services such as servers, storage, databases, "
          "networking, and software over the internet. Cloud computing allows organizations to access "
          "computing resources on demand without maintaining physical hardware. Common cloud computing "
          "models include infrastructure as a service, platform as a service, and software as a "
          "service. Many modern applications, including data science and machine learning systems, "
          "rely on cloud computing resources.",

    "D8": "Cybersecurity refers to the practice of protecting computer systems, networks, and data from "
          "unauthorized access, attacks, and damage. Cybersecurity has become increasingly important "
          "due to the growing number of cyber threats targeting individuals and organizations. Common "
          "cybersecurity measures include firewalls, encryption, and access control mechanisms. As more "
          "services move to cloud computing, cybersecurity plays a critical role in protecting "
          "sensitive data."
}

print(f"Document collection created with {len(documents)} documents.")


Document collection created with 8 documents.


## 3. Adding Documents

A single function to add new documents to the collection. Note that adding a document only updates the raw text — the dictionary and inverted index must be rebuilt afterward (this is demonstrated in Section 10).

In [ ]:
def add_document(doc_id, text):
    """Add a new document to the collection."""
    documents[doc_id] = text
    print(f"Document \'{doc_id}\' added to the collection.")


## 4. Text Preprocessing

Converts text to a clean list of terms: lowercase → strip punctuation → split into tokens → drop stopwords and very short tokens. Stemming/lemmatization is intentionally skipped — for a collection this small it adds complexity without changing which documents match, so it isn't worth the extra complexity.

In [ ]:
# A small, hand-picked stopword list instead of relying on an NLP library
STOPWORDS = {
    "a", "an", "the", "and", "or", "not", "is", "are", "was", "were", "be", "been", "being",
    "in", "on", "at", "to", "for", "of", "with", "by", "from", "as", "that", "this", "these", "those",
    "it", "its", "their", "they", "he", "she", "we", "you", "i", "but", "if", "then", "than", "so",
    "such", "can", "could", "will", "would", "may", "might", "must", "shall", "should",
    "do", "does", "did", "have", "has", "had", "which", "who", "whom", "what", "when", "where", "why", "how",
    "into", "over", "under", "between", "about", "also", "more", "most", "other", "some", "many", "much",
    "used", "using", "use"
}

def preprocess(text):
    """Convert raw text into a list of cleaned, meaningful terms."""
    text = text.lower()                        # Step 1: normalize case
    text = re.sub(r'[^a-z0-9\s]', ' ', text)    # Step 2: remove punctuation
    tokens = text.split()                       # Step 3: split into words
    terms = [t for t in tokens if t not in STOPWORDS and len(t) > 1]  # Step 4: remove stopwords
    return terms

# Quick sanity check
print(preprocess(documents["D1"]))


['artificial', 'intelligence', 'branch', 'computer', 'science', 'focuses', 'building', 'systems', 'capable', 'performing', 'tasks', 'normally', 'require', 'human', 'intelligence', 'tasks', 'include', 'reasoning', 'problem', 'solving', 'perception', 'language', 'understanding', 'modern', 'artificial', 'intelligence', 'systems', 'often', 'rely', 'machine', 'learning', 'techniques', 'improve', 'performance', 'time', 'applications', 'artificial', 'intelligence', 'found', 'robotics', 'healthcare', 'finance', 'fields']


## 5. Building the Dictionary

Collects every unique term across the whole (preprocessed) collection into a single vocabulary set.

In [ ]:
dictionary = set()

def build_dictionary():
    """Build the vocabulary (dictionary) of all unique terms in the collection."""
    global dictionary
    dictionary = set()
    for doc_id, text in documents.items():
        terms = preprocess(text)
        dictionary.update(terms)
    return dictionary

build_dictionary()
print(f"Total unique terms in dictionary: {len(dictionary)}")
print(sorted(dictionary))


Total unique terms in dictionary: 204
['access', 'accessed', 'algorithms', 'allow', 'allows', 'analysis', 'analyze', 'any', 'applications', 'artificial', 'attacks', 'based', 'become', 'big', 'branch', 'build', 'building', 'capable', 'classification', 'cloud', 'collection', 'combines', 'common', 'computer', 'computers', 'computing', 'concerned', 'control', 'core', 'create', 'critical', 'cyber', 'cybersecurity', 'damage', 'data', 'database', 'databases', 'datasets', 'decisions', 'delete', 'delivery', 'demand', 'documents', 'domain', 'due', 'efficient', 'efficiently', 'electronically', 'elements', 'enables', 'encryption', 'engines', 'essential', 'explicitly', 'extract', 'field', 'fields', 'finance', 'firewalls', 'focuses', 'found', 'generate', 'growing', 'growth', 'hardware', 'healthcare', 'human', 'image', 'important', 'improve', 'include', 'including', 'increased', 'increasingly', 'index', 'indexing', 'individuals', 'information', 'infrastructure', 'insights', 'intelligence', 'interacti

## 6. Building the Inverted Index

For each term, stores the set of document IDs it appears in. This is the core data structure of the whole system.

In [ ]:
inverted_index = {}

def build_inverted_index():
    """Build an inverted index mapping each term to the set of documents containing it."""
    global inverted_index
    inverted_index = {}
    for doc_id, text in documents.items():
        terms = preprocess(text)
        for term in terms:
            if term not in inverted_index:
                inverted_index[term] = set()
            inverted_index[term].add(doc_id)
    return inverted_index

build_inverted_index()

print("Inverted Index:")
for term in sorted(inverted_index):
    print(f"{term:15} -> {sorted(inverted_index[term])}")


Inverted Index:
access          -> ['D7', 'D8']
accessed        -> ['D6']
algorithms      -> ['D2', 'D3']
allow           -> ['D5', 'D6']
allows          -> ['D7']
analysis        -> ['D5']
analyze         -> ['D3']
any             -> ['D4']
applications    -> ['D1', 'D4', 'D6', 'D7']
artificial      -> ['D1', 'D2', 'D5']
attacks         -> ['D8']
based           -> ['D2', 'D4']
become          -> ['D8']
big             -> ['D3']
branch          -> ['D1']
build           -> ['D2']
building        -> ['D1', 'D4']
capable         -> ['D1']
classification  -> ['D5']
cloud           -> ['D7', 'D8']
collection      -> ['D4', 'D6']
combines        -> ['D3']
common          -> ['D2', 'D4', 'D5', 'D7', 'D8']
computer        -> ['D1', 'D2', 'D3', 'D8']
computers       -> ['D5']
computing       -> ['D7', 'D8']
concerned       -> ['D5']
control         -> ['D8']
core            -> ['D4']
create          -> ['D6']
critical        -> ['D8']
cyber           -> ['D8']
cybersecurity   -> ['D8']
damage

## 7. Boolean Retrieval

Implements AND, OR, NOT using plain set operations, which is exactly how Boolean retrieval works conceptually.

In [ ]:
def normalize_term(term):
    """Clean a single query term the same way documents are cleaned."""
    term = term.lower()
    term = re.sub(r'[^a-z0-9]', '', term)
    return term

def boolean_and(term1, term2):
    """Documents containing BOTH term1 and term2."""
    docs1 = inverted_index.get(normalize_term(term1), set())
    docs2 = inverted_index.get(normalize_term(term2), set())
    return docs1 & docs2

def boolean_or(term1, term2):
    """Documents containing EITHER term1 or term2."""
    docs1 = inverted_index.get(normalize_term(term1), set())
    docs2 = inverted_index.get(normalize_term(term2), set())
    return docs1 | docs2

def boolean_not(term):
    """Documents that do NOT contain the term."""
    docs_with_term = inverted_index.get(normalize_term(term), set())
    all_docs = set(documents.keys())
    return all_docs - docs_with_term

def search_term(term):
    """Documents containing a single term (used when no AND/OR/NOT is given)."""
    return inverted_index.get(normalize_term(term), set())


## 8. Search Results

`boolean_search` handles the three simple query patterns the assignment asks for (`X AND Y`, `X OR Y`, `NOT X`, or a bare term). `display_results` formats the output.

In [ ]:
def boolean_search(query):
    """Parse a simple Boolean query and return the matching document IDs."""
    tokens = query.strip().split()
    upper_tokens = [t.upper() for t in tokens]

    if "AND" in upper_tokens:
        idx = upper_tokens.index("AND")
        return boolean_and(tokens[idx - 1], tokens[idx + 1])
    elif "OR" in upper_tokens:
        idx = upper_tokens.index("OR")
        return boolean_or(tokens[idx - 1], tokens[idx + 1])
    elif "NOT" in upper_tokens:
        idx = upper_tokens.index("NOT")
        return boolean_not(tokens[idx + 1])
    else:
        return search_term(tokens[0])

def display_results(query):
    """Run a query and print the matching documents in a readable format."""
    result_ids = boolean_search(query)
    print(f"Query: {query}")
    print("Matching Documents:")
    if not result_ids:
        print("No documents found.")
    else:
        for doc_id in sorted(result_ids):
            print(f"\n{doc_id}:")
            print(documents[doc_id])
    print("-" * 70)


## 9. Testing the System

Covers AND, OR, NOT, and a non-existent term. The add-document workflow is tested in Section 10.

In [ ]:
print("TEST 1: AND query")
display_results("machine AND learning")

print("TEST 2: OR query")
display_results("cloud OR database")

print("TEST 3: NOT query")
display_results("NOT cybersecurity")

print("TEST 4: term that does not exist yet")
display_results("quantum")


TEST 1: AND query
Query: machine AND learning
Matching Documents:

D1:
Artificial intelligence is a branch of computer science that focuses on building systems capable of performing tasks that normally require human intelligence. These tasks include reasoning, problem solving, perception, and language understanding. Modern artificial intelligence systems often rely on machine learning techniques to improve their performance over time. Applications of artificial intelligence can be found in robotics, healthcare, finance, and many other fields.

D2:
Machine learning is a subset of artificial intelligence that enables computer systems to learn patterns from data without being explicitly programmed. Machine learning algorithms build a model based on sample data, known as training data, in order to make predictions or decisions. Common types of machine learning include supervised learning, unsupervised learning, and reinforcement learning. Machine learning is widely used in recommendation s

## 10. Adding a New Document

Demonstrates the required "add document → rebuild index → search again" flow.

In [ ]:
add_document(
    "D9",
    "Quantum computing uses principles of quantum mechanics such as superposition and entanglement "
    "to process information in fundamentally new ways. Unlike classical computers, quantum computers "
    "use quantum bits or qubits that can represent multiple states simultaneously. Quantum computing "
    "has the potential to solve certain complex problems much faster than traditional computers. "
    "Researchers are exploring applications of quantum computing in cryptography, optimization, "
    "and simulation."
)

# The dictionary and inverted index must be rebuilt after any change to the collection
build_dictionary()
build_inverted_index()
print(f"Dictionary and inverted index rebuilt. Total terms now: {len(dictionary)}")

print("TEST 5: searching for a term from the newly added document")
display_results("quantum")


Document 'D9' added to the collection.
Dictionary and inverted index rebuilt. Total terms now: 230
TEST 5: searching for a term from the newly added document
Query: quantum
Matching Documents:

D9:
Quantum computing uses principles of quantum mechanics such as superposition and entanglement to process information in fundamentally new ways. Unlike classical computers, quantum computers use quantum bits or qubits that can represent multiple states simultaneously. Quantum computing has the potential to solve certain complex problems much faster than traditional computers. Researchers are exploring applications of quantum computing in cryptography, optimization, and simulation.
----------------------------------------------------------------------


## 11. Interactive Search

Type a Boolean query such as `machine AND learning`, `cloud OR database`, or `NOT cybersecurity`. Type `exit` to stop.

In [ ]:
print("Type a Boolean query, e.g. \'machine AND learning\', \'cloud OR database\', \'NOT cybersecurity\'.")
print("Type \'exit\' to stop.\n")

while True:
    user_query = input("Enter query: ")
    if user_query.strip().lower() == "exit":
        break
    display_results(user_query)


Type a Boolean query, e.g. 'machine AND learning', 'cloud OR database', 'NOT cybersecurity'.
Type 'exit' to stop.

Enter query: cloud OR database
Query: cloud OR database
Matching Documents:

D6:
A database is an organized collection of structured data that is stored and accessed electronically. Databases are managed using database management systems that allow users to create, read, update, and delete data efficiently. Relational databases store data in tables and use structured query language to manage the data. Databases are essential for storing information used by websites, applications, and information retrieval systems.

D7:
Cloud computing is the delivery of computing services such as servers, storage, databases, networking, and software over the internet. Cloud computing allows organizations to access computing resources on demand without maintaining physical hardware. Common cloud computing models include infrastructure as a service, platform as a service, and software as a s